# Data understanding over the original datasets


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import seaborn as sns
import uproot

AttributeError: 'RcParams' object has no attribute '_get'

In [8]:
DATASETS = {
    "ZZTo2L2Nu": "../SkimmedDatasets/skimmed_ZZTo2L2Nu.root",
    "ZZZ": "../SkimmedDatasets/skimmed_ZZZ.root",
    "HToAATo2Mu2B": "../SkimmedDatasets/skimmed_HToAATo2Mu2B.root",
}

TREE_NAME = "Events"

FLOAT_FEATURES = [
    "MET_pt", "MET_phi", "MET_covXX", "MET_covXY", "MET_covYY",
    "MET_significance", "GenMET_pt",
    "PV_chi2", "PV_score", "PV_x", "PV_y", "PV_z",
]
INT_FEATURES = ["nSV", "nElectron", "nMuon", "nJet", "nGenJet"]

PALETTE = ["#4C72B0", "#DD8452", "#55A868"]

In [9]:
def _load_dataset(path: str, tree_name: str) -> pd.DataFrame:
    """
        Loads branches from a .root file and returns a Pandas DataFrame.
    """
    all_branches = FLOAT_FEATURES + INT_FEATURES
    with uproot.open(f"{path}:{tree_name}") as tree:
        df = tree.arrays(all_branches, library="pd")

    # Explicit casting
    df[FLOAT_FEATURES] = df[FLOAT_FEATURES].astype(np.float64)
    df[INT_FEATURES]   = df[INT_FEATURES].astype(np.int64)
    return df

In [ ]:
def _plot_correlation_heatmap(data, label, color):
    corr = df[FLOAT_FEATURES].replace([np.inf, -np.inf], np.nan).corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        corr,
        annot=True, fmt=".2f",
        #xticklabels=FLOAT_FEATURES, yticklabels=FLOAT_FEATURES,
        cmap="coolwarm", linewidths=0.5, linecolor='gray', annot_kws={"size": 8})
    plt.title("Correlation Heatmap - {label}")
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()#rect=[0, 0.03, 1, 1])
    plt.show()
    return

In [ ]:
def _plot_genmet_vs_met(datasets):
    fig, ax = plt.subplots(figsize=(7, 6))

    for (label, data), color in zip(datasets.items(), PALETTE):
        x = data["MET_pt"]
        y = data["GenMET_pt"]
        mask = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[mask], y[mask], s=4, alpha=0.35, color=color, label=label, rasterized=True)

    # linea di riferimento y = x
    lim_max = ax.get_xlim()[1]
    lim = [0, lim_max]
    ax.plot(lim, lim, "k--", lw=1, label="y = x")

    ax.set_xlabel("MET_pt [GeV]", fontsize=11)
    ax.set_ylabel("GenMET_pt [GeV]", fontsize=11)
    ax.set_title("GenMET vs MET", fontsize=13, fontweight="bold")
    ax.legend(markerscale=3, fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.plot()
    return

In [ ]:
def _data_distributions(datasets):
    n_feat = len(FLOAT_FEATURES)
    ncols = 3
    nrows = (n_feat + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    for idx, feat in enumerate(FLOAT_FEATURES):
        ax = axes[idx]
        for (label, data), color in zip(datasets.items(), PALETTE):
            vals = data[feat]
            vals = vals[np.isfinite(vals)]
            # range comune su tutti i dataset per questa feature
            ax.hist(vals, bins=60, density=True, histtype="step",
                    linewidth=1.5, color=color, label=label)
        ax.set_xlabel(feat, fontsize=9)
        ax.set_ylabel("Density", fontsize=8)
        ax.set_title(feat, fontsize=9, fontweight="bold")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.25)

    # nascondi assi vuoti
    for j in range(idx + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Float Feature Distributions", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    plt.plot()
    return

In [ ]:
def _plot_int_distributions(datasets):
    # Feature singole (escluse nJet/nGenJet che vanno insieme)
    solo_features = ["nSV", "nElectron", "nMuon"]
    paired = ("nJet", "nGenJet")

    n_plots = len(solo_features) + 1  # +1 per il plot combinato nJet/nGenJet
    ncols = 2
    nrows = (n_plots + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows))
    axes = axes.flatten()

    def draw_bar(ax, feat, datasets, colors, label_suffix=""):
        all_vals = np.concatenate([data[feat] for data in datasets.values()])
        bins = np.arange(int(all_vals.min()), int(all_vals.max()) + 2) - 0.5
        centers = (bins[:-1] + bins[1:]) / 2
        width = 0.8 / len(datasets)
        for i, ((label, data), color) in enumerate(zip(datasets.items(), colors)):
            counts, _ = np.histogram(data[feat], bins=bins)
            offset = (i - len(datasets) / 2 + 0.5) * width
            ax.bar(centers + offset, counts, width=width * 0.9,
                   color=color, label=f"{label}{label_suffix}", alpha=0.85)

    # Feature singole
    for idx, feat in enumerate(solo_features):
        ax = axes[idx]
        draw_bar(ax, feat, datasets, PALETTE)
        ax.set_xlabel(feat, fontsize=10)
        ax.set_ylabel("Counts", fontsize=10)
        ax.set_title(feat, fontsize=11, fontweight="bold")
        ax.legend(fontsize=8)
        ax.grid(axis="y", alpha=0.3)

    # nJet vs nGenJet — plot combinato
    ax_pair = axes[len(solo_features)]
    dataset_colors_ext = PALETTE + [c + "80" for c in PALETTE]  # trasparenza per nGenJet

    # costruisci array globale per i bin
    all_jet = np.concatenate(
        [np.concatenate([data["nJet"], data["nGenJet"]]) for data in datasets.values()]
    )
    bins = np.arange(int(all_jet.min()), int(all_jet.max()) + 2) - 0.5
    centers = (bins[:-1] + bins[1:]) / 2
    n_groups = len(datasets) * 2  # dataset × (nJet, nGenJet)
    width = 0.8 / n_groups

    bar_idx = 0
    legend_handles = []
    for (label, data), color in zip(datasets.items(), PALETTE):
        for feat, hatch, ls in [("nJet", "", "-"), ("nGenJet", "//", "--")]:
            counts, _ = np.histogram(data[feat], bins=bins)
            offset = (bar_idx - n_groups / 2 + 0.5) * width
            b = ax_pair.bar(
                centers + offset, counts, width=width * 0.9,
                color=color, alpha=0.7 if feat == "nJet" else 0.4,
                hatch=hatch, edgecolor="grey", linewidth=0.4,
                label=f"{label} — {feat}",
            )
            legend_handles.append(b)
            bar_idx += 1

    ax_pair.set_xlabel("n", fontsize=10)
    ax_pair.set_ylabel("Counts", fontsize=10)
    ax_pair.set_title("nJet vs nGenJet", fontsize=11, fontweight="bold")
    ax_pair.legend(fontsize=7, ncol=2)
    ax_pair.grid(axis="y", alpha=0.3)

    # nascondi assi vuoti
    for j in range(len(solo_features) + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Integer Feature Distributions", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    plt.plot()
    return

---

In [ ]:
datasets = {}
    for label, path in DATASETS.items():
        print(f"Caricamento {label}: {path}")
        try:
            datasets[label] = load_dataset(path, TREE_NAME)
            n = len(next(iter(datasets[label].values())))
            print(f"  → {n:,} eventi caricati")
        except Exception as e:
            print(f"  ERRORE: {e}")

In [11]:
label = list(DATASETS.items())[0][0]
path = DATASETS.get("ZZTo2L2Nu")

df = _load_dataset(path, TREE_NAME)
print(f"Dataset {label}")
print(f"    Shape: {df.shape}[0] features x {df.shape}[1] events")

NameError: name 'uproot' is not defined